# QAMP Q1: collect contract data

Collects data for the tokens in `rwa-quantumclock/data/sample_tokens.txt`: source code, controllers,
role members, governance events, contract code, and the same address on 12 other chains.
It only collects; `python -m qamp facts` interprets the data later.

**Steps**
1. Check `REPO_REF` in Cell 1 (the tag that fixes the token list).
2. *Runtime → Run all*. Paste a **new** Etherscan key when asked (it is hidden and not saved).
3. It takes about an hour for 100 tokens. If it stops, run Cell 2 again; finished tokens are skipped.
4. `qamp_q1_data.zip` downloads at the end. Unzip it into `rwa-quantumclock/data/` and commit.
5. Delete the key on etherscan.io/apidashboard.

Chains your key cannot read show up as errors. That is expected.


In [ ]:
# Cell 1: setup (only edit REPO_REF)
REPO_URL = "https://github.com/FahdSaif/RWA-Asset-tokenization.git"
REPO_REF = "quantumclock-q1-plan"      # tag that fixes the token list

import subprocess, sys, os
if not os.path.exists("repo"):
    subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_REF, REPO_URL, "repo"], check=True)
sys.path.insert(0, "repo/rwa-quantumclock/src")
commit = subprocess.run(["git", "-C", "repo", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
print("plan tag", REPO_REF, "commit", commit[:10])


In [ ]:
# Cell 2: run (nothing to edit; safe to run again)
import datetime as dt, getpass, json, zipfile
from pathlib import Path
import requests
from qamp.extraction import Observer, digest

def read_sample(p):
    return [l.split()[0].lower() for l in Path(p).read_text().splitlines() if l.strip() and not l.startswith("#")]

TOKENS = read_sample("repo/rwa-quantumclock/data/sample_tokens.txt")
print(len(TOKENS), "tokens in the fixed sample")
out = Path("qamp_q1")
for d in ("sources", "bytecode", "tokens"):
    (out / d).mkdir(parents=True, exist_ok=True)

key = getpass.getpass("Paste your Etherscan API key (hidden): ").strip()
def get_json(u):
    try:
        r = requests.get(u, timeout=60); r.raise_for_status(); return r.json()
    except requests.RequestException as e:
        raise RuntimeError(f"network error: {type(e).__name__}") from None   # hide the URL: it contains the key

ob = Observer(get_json, key)
block = ob.block()
for i, a in enumerate(TOKENS, 1):
    f = out / "tokens" / f"{a}.json"
    if f.exists():
        continue
    rec = ob.observe_token(a)
    for k, suffix in (("source", ""), ("implementation_source", ".impl")):
        if rec.get(k):
            (out / "sources" / f"{a}{suffix}.sol.txt").write_text(rec[k])
            rec[k] = f"sources/{a}{suffix}.sol.txt"
    f.write_text(json.dumps(rec))
    n_chains = sum(1 for c in rec["chains"].values() if c.get("code"))
    print(f"{i:3d} {a[:10]}... {str(rec['contract_name'])[:20]:20s} proxy={rec['is_proxy']!s:5} ctrl={len(rec['controllers'])} "
          f"roles={len(rec['roles'])} events={len(rec['events'])} other_chains={n_chains} errors={len(rec['errors'])}")
del key

records = [json.loads((out / "tokens" / f"{a}.json").read_text()) for a in TOKENS]
for rec in records:                     # also save contract code as files
    for k in ("token", "implementation"):
        code = (rec.get("bytecode") or {}).get(k)
        if isinstance(code, str) and code.startswith("0x"):
            (out / "bytecode" / f"{rec['address']}.{k}.hex").write_text(code)
retrieved = dt.datetime.now(dt.timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")
raw = dict(tokens=records, block=block, retrieved_at_utc=retrieved, plan_ref=REPO_REF, plan_commit=commit)
(out / "q1_raw.json").write_text(json.dumps(raw))
summary = dict(n_tokens=len(records), with_errors=sum(bool(r["errors"]) for r in records), block=block,
               retrieved_at_utc=retrieved, plan_ref=REPO_REF, plan_commit=commit, sha256_raw=digest(raw))
(out / "q1_summary.json").write_text(json.dumps(summary, indent=1))
print(json.dumps(summary, indent=1))
with zipfile.ZipFile("qamp_q1_data.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for p in out.rglob("*"):
        if "tokens" not in p.parts:
            z.write(p, p.relative_to(out))
try:
    from google.colab import files; files.download("qamp_q1_data.zip")
except ImportError:
    print("wrote qamp_q1_data.zip")
